# Calculate VIX shocks

This notebook loads `C:/Python/CSUREMM/data_primitive/vix.csv`, constructs VIX shock measures, and writes the outputs to the same output directory used by the S&P 500 workflow.

The notebook is designed to be run after the S&P 500 workflow directory structure exists. If your S&P 500 workflow uses a different output folder, edit `OUTPUT_DIR` in the configuration cell.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
INPUT_PATH = Path(r"C:/Python/CSUREMM/data_primitive/vix.csv")

# Use the same output directory as the S&P 500 workflow.
# Edit this line if your S&P 500 workflow uses a different folder.
OUTPUT_DIR = Path(r"C:/Python/CSUREMM/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", INPUT_PATH)
print("Output directory:", OUTPUT_DIR)

## Load and standardize the VIX series

The code below accepts common date/value column names. It renames the date column to `date` and the VIX level column to `vix`.

In [ ]:
def load_vix(input_path: Path) -> pd.DataFrame:
    df = pd.read_csv(input_path)

    # Detect date column.
    date_candidates = [
        "date", "Date", "DATE", "time", "Time", "TIME", "observation_date"
    ]
    date_col = next((c for c in date_candidates if c in df.columns), None)
    if date_col is None:
        date_col = df.columns[0]

    # Detect VIX value column.
    value_candidates = [
        "vix", "VIX", "close", "Close", "CLOSE", "value", "Value", "VALUE", "Adj Close", "Adj_Close"
    ]
    value_col = next((c for c in value_candidates if c in df.columns and c != date_col), None)
    if value_col is None:
        non_date_cols = [c for c in df.columns if c != date_col]
        if len(non_date_cols) != 1:
            raise ValueError(
                "Could not infer the VIX value column. "
                f"Columns found: {list(df.columns)}. Rename the VIX column to 'vix'."
            )
        value_col = non_date_cols[0]

    out = (
        df[[date_col, value_col]]
        .rename(columns={date_col: "date", value_col: "vix"})
        .copy()
    )
    out["date"] = pd.to_datetime(out["date"])
    out["vix"] = pd.to_numeric(out["vix"], errors="coerce")
    out = (
        out
        .dropna(subset=["date", "vix"])
        .sort_values("date")
        .drop_duplicates("date", keep="last")
        .reset_index(drop=True)
    )
    return out

vix = load_vix(INPUT_PATH)
vix.head(), vix.tail(), vix.shape

## Compute VIX shocks

Definitions used here:

- `vix_change`: daily first difference in VIX level.
- `vix_pct_change`: daily percent change in VIX level.
- `vix_log_change`: daily log change in VIX level.
- `vix_change_z`: expanding-standardized daily VIX change, using only information available up to that date.
- `vix_shock_pos_1sd`: positive shock indicator where `vix_change_z >= 1`.
- `vix_shock_pos_2sd`: large positive shock indicator where `vix_change_z >= 2`.
- `vix_shock_neg_1sd`: negative shock indicator where `vix_change_z <= -1`.
- `vix_shock_abs_2sd`: large absolute shock indicator where `abs(vix_change_z) >= 2`.

In [ ]:
def calculate_vix_shocks(df: pd.DataFrame, min_periods: int = 60) -> pd.DataFrame:
    out = df.copy()

    out["vix_change"] = out["vix"].diff()
    out["vix_pct_change"] = out["vix"].pct_change()
    out["vix_log_change"] = np.log(out["vix"]).diff()

    # Expanding mean/std shifted by one day to avoid look-ahead bias.
    expanding_mean = out["vix_change"].expanding(min_periods=min_periods).mean().shift(1)
    expanding_std = out["vix_change"].expanding(min_periods=min_periods).std(ddof=1).shift(1)

    out["vix_change_z"] = (out["vix_change"] - expanding_mean) / expanding_std

    out["vix_shock_pos_1sd"] = (out["vix_change_z"] >= 1).astype("Int64")
    out["vix_shock_pos_2sd"] = (out["vix_change_z"] >= 2).astype("Int64")
    out["vix_shock_neg_1sd"] = (out["vix_change_z"] <= -1).astype("Int64")
    out["vix_shock_abs_2sd"] = (out["vix_change_z"].abs() >= 2).astype("Int64")

    return out

vix_shocks = calculate_vix_shocks(vix, min_periods=60)
vix_shocks.head(70).tail(10)

## Save outputs

The notebook writes a full shock dataset and a compact event-only file to `OUTPUT_DIR`.

In [ ]:
full_output_path = OUTPUT_DIR / "vix_shocks.csv"
event_output_path = OUTPUT_DIR / "vix_shock_events.csv"
summary_output_path = OUTPUT_DIR / "vix_shock_summary.csv"

vix_shocks.to_csv(full_output_path, index=False)

vix_events = vix_shocks.loc[
    vix_shocks[[
        "vix_shock_pos_1sd",
        "vix_shock_pos_2sd",
        "vix_shock_neg_1sd",
        "vix_shock_abs_2sd",
    ]].fillna(0).any(axis=1)
].copy()
vix_events.to_csv(event_output_path, index=False)

summary = pd.DataFrame({
    "metric": [
        "n_observations",
        "start_date",
        "end_date",
        "n_pos_1sd_shocks",
        "n_pos_2sd_shocks",
        "n_neg_1sd_shocks",
        "n_abs_2sd_shocks",
    ],
    "value": [
        len(vix_shocks),
        vix_shocks["date"].min(),
        vix_shocks["date"].max(),
        int(vix_shocks["vix_shock_pos_1sd"].sum(skipna=True)),
        int(vix_shocks["vix_shock_pos_2sd"].sum(skipna=True)),
        int(vix_shocks["vix_shock_neg_1sd"].sum(skipna=True)),
        int(vix_shocks["vix_shock_abs_2sd"].sum(skipna=True)),
    ]
})
summary.to_csv(summary_output_path, index=False)

print("Saved:")
print(full_output_path)
print(event_output_path)
print(summary_output_path)
summary

## Optional: quick diagnostic plot

In [ ]:
import matplotlib.pyplot as plt

plot_df = vix_shocks.dropna(subset=["vix_change_z"])

plt.figure(figsize=(12, 5))
plt.plot(plot_df["date"], plot_df["vix_change_z"])
plt.axhline(2, linestyle="--")
plt.axhline(1, linestyle="--")
plt.axhline(-1, linestyle="--")
plt.axhline(-2, linestyle="--")
plt.title("Standardized VIX shocks")
plt.xlabel("Date")
plt.ylabel("VIX change z-score")
plt.tight_layout()
plt.show()